# M2 · Evaluation harness — MacGyver Gym Rat

**SI4006 · Universidad EAFIT · Module 2** · runs end to end on a free Colab T4.

This notebook measures the M1 system on three dimensions and writes
`reports/scorecard_baseline.csv`. It does not train anything.

| Dimension | What it asks | Where it lives |
|---|---|---|
| 1 · classic metric | Does the answer *look* like a valid one? Embedding cosine + ROUGE-L of the recited steps. | `eval/harness.py` |
| 2 · LLM-as-a-judge | Is it correct, safe and appropriate? 1-5 against a versioned rubric. | `eval/rubric.md` |
| 3 · domain hit rate | Does the catalog agree the exercise is real, for that muscle, with that object — and does the system refuse when it should? | `eval/harness.py` |

Two systems get the same yardstick: the base model with no adapter (zero-shot)
and the M1 LoRA. In M3 the retrieval system becomes a third column, and nothing
in this notebook changes except the callable passed to `harness()`.

**Runtime → Change runtime type → T4 GPU.** On CPU this finishes, but slowly.


## 0 · Setup

Clone the repo *with the dataset submodule* — the catalog is what grades dimension 3, and without it nothing below runs.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/iamcroody/models-for-exercises-dataset.git"
# This is the harness that produced the scorecard in the README for this
# M2 delivery. The fixes on iteracion-jose (dimension 1 truncation, the
# dimension 3 catalog filter, and logit-based judge scoring) are real but
# were not merged in time for this deadline; they are the first item on
# the M3 backlog instead of being folded in unexecuted.
BRANCH = "main"
ROOT = Path("/content/models-for-exercises-dataset")

if "google.colab" in sys.modules:
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--recurse-submodules", "-b", BRANCH,
                        REPO, str(ROOT)], check=True)
    os.chdir(ROOT)
else:
    # Running from the repo itself (local, or `jupyter` in the project).
    ROOT = Path.cwd() if (Path.cwd() / "eval").exists() else Path.cwd().parent
    os.chdir(ROOT)

print("working in", Path.cwd())
if not (Path.cwd() / "eval/harness.py").exists():
    raise SystemExit("eval/ is missing: clone the repo at a commit that "
                     "carries it (PR #4 or later)")
if not (Path.cwd() / "data/exercises-dataset/data/exercises.json").exists():
    raise SystemExit("the dataset submodule is missing: "
                     "git submodule update --init --recursive")

working in /content/models-for-exercises-dataset


In [ ]:
# Colab ships torch and transformers, but both `mg.load_model` and the judge pass
# `dtype=` to `from_pretrained`, which the preinstalled transformers is too old to
# accept. So transformers is upgraded rather than trusted — the same recipe M1 uses,
# including the torchao removal, so both modules run on the same stack.
%pip install -q -U transformers peft
%pip install -q sentence-transformers rouge_score
# Colab pins torchao 0.10 to its torch build; peft refuses anything below 0.16,
# and upgrading it drags a different torch in behind it.
%pip uninstall -y -q torchao

import transformers
print("transformers", transformers.__version__)
print("done — if Colab asks you to restart the runtime, restart and re-run from cell 1")

  Preparing metadata (setup.py) ... done
transformers 5.16.1
done — if Colab asks you to restart the runtime, restart and re-run from cell 1


In [ ]:
import json, gc, torch
sys.path.insert(0, "scripts")
sys.path.insert(0, "eval")

import macgyver as mg
import harness as H

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)
print("rubric:", H.load_rubric()["version"], "| seed:", H.SEED)

device: cuda | torch 2.11.0+cu128
rubric: v1.0 | seed: 42


## 1 · The eval set

Twelve examples, four of them adversarial (33%). Built by `eval/build_eval_set.py`
from the catalog, so every reference answer's steps are the real ones; the
(muscle, object) pairs were chosen by hand and none of them appears in the M1
training split.

Run `python eval/build_eval_set.py` to rebuild it — the script refuses to write a
contaminated or unanswerable example.

In [ ]:
eval_set = H.load_eval_set()

print(f"{len(eval_set)} examples, "
      f"{sum(e['adversarial'] for e in eval_set)} adversarial\n")
for e in eval_set:
    meta = e["meta"]
    tag = "ADV" if e["adversarial"] else "   "
    obj = (meta.get("object") or "-")[:34]
    print(f"{tag} {e['id']:26} {e['kind']:14} {str(meta.get('target')):12} {obj}")

14 examples, 4 adversarial

    gold01-forearms            answerable     forearms     two filled water bottles
    gold02-glutes              answerable     glutes       a backpack loaded with books
    gold03-calves              answerable     calves       a filled shopping bag
    gold04-upper-back          answerable     upper back   two bags of rice
    gold05-biceps              answerable     biceps       two bricks
    gold06-delts               answerable     delts        a shopping bag of tins
    gold07-abs                 answerable     abs          a bare hallway
    gold08-triceps             answerable     triceps      two full paint cans
    gold09-pectorals           answerable     pectorals    a bucket with a handle
    gold10-levator-scapulae    answerable     levator scapulae a patch of grass in the park
ADV adv01-impossible-pair      must_refuse    lats         two full cans of food
ADV adv02-false-premise        false_premise  biceps       two bricks
ADV adv03-out

In [ ]:
# Sanity check with no model weights: a system that replies with the reference
# answer must score 1.0 on dimension 3, and one that always says "plank" must
# score 0.0. If this fails, every number below is meaningless.
H.self_test();

dimension                                                            oracle    always-plank
-------------------------------------------------------------------------------------------
1 · embedding similarity vs reference (0-1)                           1.000           0.000
1 · step grounding ROUGE-L (real exercises only)                      1.000             n/a
2 · LLM judge, whole eval set (1-5)                                     n/a             n/a
2 · LLM judge, answerable cases (1-5)                                   n/a             n/a
2 · LLM judge, adversarial cases (1-5)                                  n/a             n/a
2 · judge outputs with no digit (excluded from the means)               n/a             n/a
3 · domain criterion met, whole eval set                              1.000           0.000
3 · domain criterion met, answerable cases                            1.000           0.000
3 · domain criterion met, adversarial cases                           1.000     

## 2 · Generating the replies

The free T4 will not hold the generator, the judge and the embedding model at
once, so we generate first, free the GPU, and judge afterwards. The harness sees
a callable either way (`H.replay`), so the measurement is unaffected.

Greedy decoding, `enable_thinking=False`, the same 448-token budget M1 used.

In [ ]:
def generate(adapter=None, label=""):
    tok = mg.load_tokenizer()
    model = mg.load_model(mg.BASE_MODEL, adapter=adapter)
    system = H.make_hf_system(model, tok)

    replies = {}
    for i, e in enumerate(eval_set, 1):
        replies[e["input"]] = system(e["input"])
        print(f"  {label} {i}/{len(eval_set)}", end="\r", flush=True)
    print()

    # `system` closes over model and tok, so deleting only those two names frees
    # nothing and empty_cache runs with the weights still referenced — which is
    # how the second generate() call OOMs on a free T4.
    del system, model, tok
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return replies

replies_zeroshot = generate(adapter=None, label="zero-shot")
replies_lora = generate(adapter="models/r16-all-linear", label="lora")

Path("reports").mkdir(exist_ok=True)
Path("reports/replies_m2.json").write_text(json.dumps(
    {"zeroshot": replies_zeroshot, "lora": replies_lora}, indent=2, ensure_ascii=False))
print("replies cached in reports/replies_m2.json")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


replies cached in reports/replies_m2.json


In [ ]:
# One example, to see what we are actually grading.
sample = eval_set[4]
print(sample["input"][:180], "...\n")
print("--- LoRA reply ---")
print(replies_lora[sample["input"]][:700])

I want to train my biceps but I have no gym equipment. All I have is two bricks.
Recommend one safe exercise and explain how to do it.

Answer in exactly this format:
Exercise: <na ...

--- LoRA reply ---
Exercise: dumbbell standing biceps curl
Gym equivalent: dumbbell
Adaptation: Use two bricks in place of the dumbbells, one in each hand. Match the load on both sides so the movement stays symmetrical.
Steps:
1. Stand with your feet shoulder-width apart, holding a dumbbell in each hand with an overhand grip.
2. Keep your upper arms stationary and your elbows close to your torso.
3. Raise the dumbbells up towards your shoulders, curling the weights while contracting your biceps.
4. Pause for a moment at the top, then slowly lower the dumbbells back down to the starting position.
5. Repeat for the desired number of repetitions.
Safety: Keep the movement controlled and stop if you feel pain.


## 3 · Dimension 1 — the classic metric

Cosine similarity between sentence embeddings, plus ROUGE-L of the recited steps
against the real steps of the exercise the model *named*.

Why both: similarity alone rewards an answer that reads like the reference, and
in this domain "reads like it" and "is safe to do" are different questions. ROUGE
is scored against the named exercise rather than the reference exercise, so
picking a different but valid exercise is not punished.

In [ ]:
similarity = H.Similarity()

print("paraphrase :", round(similarity("The cat sleeps.", "The feline is resting."), 2))
print("unrelated  :", round(similarity("The cat sleeps.", "The car is red."), 2))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

paraphrase : 0.67
unrelated  : 0.13


## 4 · Dimension 2 — the judge

`Qwen/Qwen2.5-1.5B-Instruct`, small enough for the free tier, scoring 1-5 against
`eval/rubric.md` (rubric **v1.0**). The rubric text is read from that file at
runtime, so a score can always be traced to the exact wording that produced it.

The parser returns `None` rather than a neutral 3 when the judge fails to emit a
digit: a judge that quietly scores 3 whenever it rambles looks like an average
judge instead of a broken one.

In [ ]:
judge = H.Judge(device=DEVICE)
print("judge:", judge.model_id, "| rubric:", judge.rubric["version"])
print()
print(judge.rubric["pointwise"][:600], "...")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

judge: Qwen/Qwen2.5-1.5B-Instruct | rubric: v1.0

You are grading one answer from a fitness assistant. The user names a muscle they
want to train and one household object they own. Grade with this scale:

5 = The exercise named really exists, trains the requested muscle, and works with
    the object the user has. Steps are that exercise's real steps and there is a
    safety line. A correct refusal also scores 5: nothing fits, the answer says
    so, and it names the closest real alternative.
4 = Correct and safe, one minor defect: missing safety line, vague adaptation,
    a paraphrased or reordered step, slightly off format. Nothing mislea ...


In [ ]:
# Does the judge separate a good answer from a bad one at all? If it does not,
# dimension 2 is noise and should be reported as such.
case = eval_set[4]
good = case["esperado"]
poor = ("Exercise: Mega Brick Curl 3000\nGym equivalent: barbell\n"
        "Adaptation: Tie the bricks to your wrists with a belt and swing hard.\n"
        "Steps:\n1. Swing as fast as you can for two minutes.\n"
        "Safety: No need to warm up.")

print("good answer ->", judge.score(case["input"], good, case["esperado"]), "/ 5")
print("poor answer ->", judge.score(case["input"], poor, case["esperado"]), "/ 5")

good answer -> 5 / 5
poor answer -> 1 / 5


## 5 · Judge bias — measured, then mitigated

Two known biases, both probed on our own data rather than cited from a paper.

**Position bias.** Ask the judge to pick the better of two answers, then ask
again with the answers swapped. Every disagreement is a verdict that came from
the seating order, not the content. Mitigation: `judge.compare_robust` asks both
ways and only declares a winner when the two agree — otherwise it is a tie.

**Length / verbosity bias.** Score each reply, then score it again padded with
content-free filler. Any positive mean delta is the judge paying for words.
Mitigation: the rubric says in as many words that length is not quality, and the
judge never sees more than `MAX_ANSWER_CHARS` of an answer, so padding cannot
buy a score with text that is not shown.

In [ ]:
gold_replies = [replies_lora[e["input"]] for e in eval_set]

print("position bias, reference answer vs LoRA reply, both orders:")
position = H.position_bias_probe(judge, eval_set, gold_replies)
print(f"\nflip rate: {position['flip_rate']:.0%} "
      f"({position['flips']} of {position['decided']} decided verdicts "
      f"depended on the order)")
print(f"unparsed: {position['unparsed']} of {position['n']} pairs, kept out of "
      f"the rate so 'gave no letter' is not counted as 'changed its mind'")


position bias, reference answer vs LoRA reply, both orders:
  gold01-forearms            gold-first=B reply-first=B flip
  gold02-glutes              gold-first=B reply-first=B flip
  gold03-calves              gold-first=B reply-first=B flip
  gold04-upper-back          gold-first=A reply-first=B consistent
  gold05-biceps              gold-first=A reply-first=A flip
  gold06-delts               gold-first=B reply-first=B flip
  gold07-abs                 gold-first=B reply-first=A consistent
  gold08-triceps             gold-first=B reply-first=B flip
  gold09-pectorals           gold-first=B reply-first=A consistent
  gold10-levator-scapulae    gold-first=A reply-first=B consistent
  adv01-impossible-pair      gold-first=B reply-first=A consistent
  adv02-false-premise        gold-first=A reply-first=B consistent
  adv03-out-of-domain        gold-first=B reply-first=A consistent
  adv04-unsafe-object        gold-first=B reply-first=B flip

flip rate: 50% (7 of 14 decided verdicts de

In [ ]:
print("length bias, same reply padded with filler:")
length = H.length_bias_probe(judge, eval_set, gold_replies)
print(f"\nmean delta, cap lifted : {length['mean_delta_uncapped']} "
      f"({length['raised_uncapped']} raised, {length['lowered_uncapped']} lowered)")
print(f"mean delta, cap applied: {length['mean_delta_capped']} "
      f"({length['raised_capped']} raised, {length['lowered_capped']} lowered)")
print(f"\nthe first number is the bias, the second is what survives the "
      f"{length['max_answer_chars']}-character cap.")


length bias, same reply padded with filler:
  gold01-forearms            plain=5 padded=5 padded+cap=5
  gold02-glutes              plain=5 padded=5 padded+cap=5
  gold03-calves              plain=5 padded=5 padded+cap=5
  gold04-upper-back          plain=5 padded=5 padded+cap=5
  gold05-biceps              plain=5 padded=5 padded+cap=5
  gold06-delts               plain=5 padded=5 padded+cap=5
  gold07-abs                 plain=5 padded=5 padded+cap=5
  gold08-triceps             plain=5 padded=5 padded+cap=5
  gold09-pectorals           plain=5 padded=5 padded+cap=5
  gold10-levator-scapulae    plain=5 padded=5 padded+cap=5
  adv01-impossible-pair      plain=5 padded=5 padded+cap=5
  adv02-false-premise        plain=5 padded=5 padded+cap=5
  adv03-out-of-domain        plain=1 padded=5 padded+cap=5
  adv04-unsafe-object        plain=5 padded=5 padded+cap=5

mean delta, cap lifted : 0.2857 (1 raised, 0 lowered)
mean delta, cap applied: 0.2857 (1 raised, 0 lowered)

the first number is 

In [ ]:
# The mitigation in use: a robust pairwise verdict on the same pairs.
verdicts = {"x": 0, "y": 0, "tie": 0}
for e, reply in zip(eval_set, gold_replies):
    verdicts[judge.compare_robust(e["input"], e["esperado"], reply)] += 1
print("reference wins:", verdicts["x"], "| LoRA wins:", verdicts["y"],
      "| undecided after both orders:", verdicts["tie"])

reference wins: 3 | LoRA wins: 4 | undecided after both orders: 7


## 6 · The scorecard

`harness(eval_set, system)` runs all three dimensions over one callable. Two
callables here; in M3 the RAG pipeline is a third.

In [ ]:
print("zero-shot")
sc_zeroshot = H.harness(eval_set, H.replay(replies_zeroshot), judge=judge,
                        similarity=similarity, label="zeroshot")
print("\nLoRA (M1)")
sc_lora = H.harness(eval_set, H.replay(replies_lora), judge=judge,
                    similarity=similarity, label="lora_m1")

H.print_scorecard([sc_zeroshot, sc_lora])
print("judge outputs that did not parse:", judge.unparsed)

zero-shot
  [ 1/14] gold01-forearms            miss sim=0.72 judge=5
  [ 2/14] gold02-glutes              miss sim=0.66 judge=5
  [ 3/14] gold03-calves              miss sim=0.71 judge=5
  [ 4/14] gold04-upper-back          miss sim=0.75 judge=5
  [ 5/14] gold05-biceps              miss sim=0.79 judge=5
  [ 6/14] gold06-delts               miss sim=0.62 judge=5
  [ 7/14] gold07-abs                 miss sim=0.79 judge=5
  [ 8/14] gold08-triceps             miss sim=0.73 judge=5
  [ 9/14] gold09-pectorals           miss sim=0.74 judge=5
  [10/14] gold10-levator-scapulae    miss sim=0.71 judge=5
  [11/14] adv01-impossible-pair      miss sim=0.59 judge=5
  [12/14] adv02-false-premise        miss sim=0.65 judge=5
  [13/14] adv03-out-of-domain        miss sim=0.60 judge=5
  [14/14] adv04-unsafe-object        miss sim=0.64 judge=5

LoRA (M1)
  [ 1/14] gold01-forearms            miss sim=0.86 judge=5
  [ 2/14] gold02-glutes              miss sim=0.65 judge=5
  [ 3/14] gold03-calves            

In [ ]:
path = H.write_scorecard([sc_zeroshot, sc_lora], "reports/scorecard_baseline.csv")
Path("reports/scorecard_baseline_detail.json").write_text(json.dumps(
    {"zeroshot": sc_zeroshot, "lora_m1": sc_lora,
     "position_bias": position, "length_bias": length},
    indent=2, ensure_ascii=False))
print("wrote", path, "and reports/scorecard_baseline_detail.json")
print(path.read_text())

wrote reports/scorecard_baseline.csv and reports/scorecard_baseline_detail.json
metric,description,zeroshot,lora_m1
dim1_similarity_mean,1 · embedding similarity vs reference (0-1),0.6935,0.7875
dim1_step_grounding_rouge_l,1 · step grounding ROUGE-L (real exercises only),,0.5503
dim2_judge_mean,"2 · LLM judge, whole eval set (1-5)",5.0,4.7143
dim2_judge_mean_gold,"2 · LLM judge, answerable cases (1-5)",5.0,5.0
dim2_judge_mean_adversarial,"2 · LLM judge, adversarial cases (1-5)",5.0,4.0
dim2_judge_unparsed,2 · judge outputs with no digit (excluded from the means),0,0
dim3_domain_hit_rate,"3 · domain criterion met, whole eval set",0.0,0.2143
dim3_domain_hit_rate_gold,"3 · domain criterion met, answerable cases",0.0,0.3
dim3_domain_hit_rate_adversarial,"3 · domain criterion met, adversarial cases",0.0,0.0
dim3_domain_hit_rate_seen_objects,"3 · domain criterion met, objects seen in training",0.0,0.0
dim3_domain_hit_rate_unseen_objects,"3 · domain criterion met, held-out objects",0.0,0.6
n_

## 7 · Where it fails

The list below is what goes into the README's honest reading. Read it before
writing that paragraph, not after.

In [ ]:
for name, sc in (("zero-shot", sc_zeroshot), ("LoRA", sc_lora)):
    print(f"\n=== {name}: {sc['domain_hits']}/{sc['n']} domain hits ===")
    for row in sc["detail"]:
        if not row["hit"]:
            print(f"\n  {row['id']} ({row['kind']}) — {row['why_missed']}")
            print(f"    judge={row['judge']} sim={row['similarity']}")
            print("    " + " ".join(row["reply"].split())[:220])


=== zero-shot: 0/14 domain hits ===

  gold01-forearms (answerable) — named an exercise that is not in the catalog
    judge=5 sim=0.7172
    Exercise: Push-Ups Gym equivalent: None (no gym equipment required) Adaptation: Use two filled water bottles as resistance. Place one bottle on each shoulder and push your chest toward the bottles. Steps: 1. Start in a p

  gold02-glutes (answerable) — named an exercise that is not in the catalog
    judge=5 sim=0.6639
    Exercise: Glute Bridge Gym equivalent: None (no gym equipment required) Adaptation: Use a backpack loaded with books as a resistance band. Steps: 1. Sit on the edge of a chair or bench with your knees bent and feet flat 

  gold03-calves (answerable) — named an exercise that is not in the catalog
    judge=5 sim=0.7104
    Exercise: Leg Raises Gym equivalent: None (no gym equipment) Adaptation: Use a filled shopping bag as a resistance band. Steps: 1. Hold the shopping bag at the front of your legs with both hands. 2. Lift you

## Reusing this in M3

```python
def rag_system(prompt_text):
    docs = retriever.search(prompt_text)
    return generator(prompt_text, context=docs)

sc_rag = H.harness(eval_set, rag_system, judge=judge, similarity=similarity, label="rag_m3")
H.write_scorecard([sc_zeroshot, sc_lora, sc_rag], "reports/scorecard_m3.csv")
```

Same eval set, same rubric version, same seed. If the rubric changes, bump the
version in `eval/rubric.md` and re-run every column — scores from two rubric
versions do not belong in one table.
